# Inference

Open this notebook with the **repository root** as the working directory, or run `pip install -e .` once from the repo root so `nmt` imports resolve from anywhere.

In [ ]:
from pathlib import Path
import sys

_root = Path.cwd().resolve()
if not (_root / "nmt").is_dir() and (_root.parent / "nmt").is_dir():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import os
os.chdir(_root)

import torch
from nmt.config import get_config, get_weights_path
from nmt.checkpoint import load_training_checkpoint
from nmt.train import get_model, get_dataset, run_validation
from nmt.translate import translate

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
config = get_config()
train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt = get_dataset(config)
model = get_model(config, tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size()).to(device)

model_filename = get_weights_path(config, str(config["checkpoint_epoch"]))
state = load_training_checkpoint(model_filename, map_location=device)
model.load_state_dict(state["model_state_dict"])

In [ ]:
run_validation(
    model,
    val_dataloader,
    tokenizer_src,
    tokenizer_tgt,
    config["seq_len"],
    device,
    lambda msg: print(msg),
    0,
    None,
    num_examples=5,
)

In [ ]:
t = translate("ein guter Student.")
print(f"Final translation: {t}")